# Stage B pipeline - v3.4.0 + stage_b_fixes rev 2

End-to-end orchestrator for the Vietnam 30-min daytime AOD product. Two parallel methods, both shipped as standalone products in `output/st_kriging/` and `output/rf/`.

| Step | Module | Output tree |
|------|--------|-------------|
| B1   | [`kriging`](kriging.py)       | `output/st_kriging/YYYY/MM/DD/aod_*.nc` |
| B2   | [`rf_gapfill`](rf_gapfill.py) | `output/rf/YYYY/MM/DD/aod_*.nc` |
| C    | [`validate`](validate.py)     | head-to-head ST kriging vs RF |

Daytime window is data-driven per day from Stage A filenames (typical ~21 slots/day). 5-fold contiguous temporal CV for both models (folds by UTC date). `oob_score=False` for RF; uncertainty = per-tree SD across the ensemble.

## 0 - Setup

In [ ]:
import os, sys, time
from datetime import date
from pathlib import Path

os.environ.setdefault('HDF5_USE_FILE_LOCKING', 'FALSE')
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import config as cfg
import kriging    as kg
import rf_gapfill as rf
import validate   as vb

for d in (cfg.ST_KRIGING_DIR, cfg.RF_OUTPUT_DIR, cfg.MODELS_DIR,
          cfg.VALIDATION_DIR):
    d.mkdir(parents=True, exist_ok=True)

TRAIN_START, TRAIN_END = cfg.TRAIN_START, cfg.TRAIN_END
TEST_START,  TEST_END  = cfg.TEST_START,  cfg.TEST_END
RUN_START,   RUN_END   = TEST_START,     TEST_END
OVERWRITE              = False

print(f'Training : {TRAIN_START}  ->  {TRAIN_END}')
print(f'Held-out : {TEST_START}  ->  {TEST_END}')
print(f'Full run : {RUN_START}  ->  {RUN_END}')

## 1 - Step B1: Spatiotemporal kriging (Yang & Hu 2018)

Fit a **metric** variogram (single component over the space-time metric distance, post-§7.7 diagnostic) on the training partition once, then ST-krige every daytime slot in the run window.  Default target is the **(AOD − CAMS) residual**; CAMS is added back at each target cell (see `config.B1_VARIOGRAM_TARGET`).

Cells the kriger cannot estimate (empty spatiotemporal pool, solver failure, or no CAMS coverage at the target) are left as NaN — no climatology backstop, per Yang & Hu 2018.

In [ ]:
vgm = kg.fit_and_save_variogram(start=TRAIN_START, end=TRAIN_END, progress=tqdm)
print(f'Target              : {cfg.B1_VARIOGRAM_TARGET}')
print(f'Metric  var         : {vgm.metric.var:.4f}')
print(f'Metric  range_km    : {vgm.metric.len_scale:.2f}')
print(f'Metric  nugget      : {vgm.metric.nugget:.4f}')
print(f'Anisotropy k (km/h) : {vgm.k_km_per_hour:.3f}')

In [ ]:
t0 = time.time()
n_b1 = kg.run(RUN_START, RUN_END,
              vgm=vgm,
              overwrite=OVERWRITE,
              progress=tqdm)
print(f'B1 wrote {n_b1} ST-kriging NC files in {time.time() - t0:.1f}s.')

## 2 - Step B2: Per-slot Random Forest

5-fold temporal-block CV (fold by UTC date). Selection criterion: mean CV-R^2. OOB-R^2 is disabled because AOD temporal autocorrelation makes OOB optimistic.

In [ ]:
MODEL_NAME = 'rf_tuned'

best_hp, tune_results = rf.tune_rf(
    start    = TRAIN_START,
    end      = TRAIN_END,
    name     = MODEL_NAME,
    progress = tqdm,
)
print('Best hyperparameters:', best_hp)

bundle = rf.load_bundle(MODEL_NAME)
print('Training window :', bundle.training_window)
print('Hyperparameters :', bundle.hyperparams)
pd.DataFrame(tune_results[:10])

In [ ]:
pd.Series(vb.internal_consistency(bundle.metrics)).to_frame('value')

In [ ]:
n_filled = rf.fill_range(
    start     = RUN_START,
    end       = RUN_END,
    bundle    = bundle,
    overwrite = OVERWRITE,
    progress  = tqdm,
)
print(f'B2 RF wrote {n_filled} NC files.')

## 3 - Validation (Section 8.2)

Head-to-head: ST kriging vs RF on identical (slot, site) keys.

In [ ]:
pairs_rf   = vb.aeronet_pairs(TEST_START, TEST_END, candidate='rf',         blind_only=True, progress=tqdm)
pairs_krig = vb.aeronet_pairs(TEST_START, TEST_END, candidate='st_kriging', blind_only=True, progress=tqdm)

print(f'AERONET-blind pairs - RF: {len(pairs_rf)}, ST kriging: {len(pairs_krig)}')
vb.metric_panel(pairs_rf)

In [ ]:
panel = vb.compare_candidates({
    'B2_RF':      pairs_rf,
    'B1_ST_krig': pairs_krig,
})
panel

In [ ]:
# Paired skill: RF minus ST kriging on identical (slot_utc, site) keys.
vb.paired_skill(pairs_rf, pairs_krig)

In [ ]:
cov_rf = vb.coverage_audit(TEST_START, TEST_END, candidate='rf', progress=tqdm)
cov_rf

In [ ]:
vb.sso_stratified_rmse(pairs_rf)

In [ ]:
vi = vb.variable_importance(MODEL_NAME)
vi.head(15)

In [ ]:
vb.cloud_period_recovery(TEST_START, TEST_END, candidate='rf')

In [ ]:
pairs_full_rf = vb.aeronet_pairs(TEST_START, TEST_END, candidate='rf', blind_only=False, progress=tqdm)
vb.success_table(pairs_full_rf, cov_rf)

---
**Done.** Outputs:

- ST kriging product   -> `Stage_B/output/st_kriging/YYYY/MM/DD/aod_*.nc`
- RF product           -> `Stage_B/output/rf/YYYY/MM/DD/aod_*.nc`
- Trained RF bundle    -> `Stage_B/models/rf_tuned.joblib`
- Fitted variogram     -> `Stage_B/models/st_variogram.json`